# 面试问题：你会怎样设计 RAG/LLM 的语义缓存？

**一句话回答**：缓存 key 必须绑定 tenant、权限、模型、prompt template、知识快照和安全策略；先做精确命中，再在同一安全命名空间内做向量近似命中，阈值由误命中成本校准。TTL/LRU 只解决容量，不解决知识更新、跨租户泄漏、缓存投毒和并发击穿。

下面只用标准库与 NumPy 实现规范化、确定性 toy embedding、cosine 检索、阈值选择、TTL/LRU、版本失效和 single-flight 状态机。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
from collections import OrderedDict, Counter  # 导入本单元所需的依赖。
import hashlib, json, math, re, unicodedata  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED80=8001  # 计算并保存当前步骤的中间状态。
assert unicodedata.normalize("NFKC","ＡＢＣ")=="ABC"  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"a").digest()!=hashlib.sha256(b"b").digest()  # 用受控断言验证关键不变量。
assert np.dot([1.,0.],[1.,0.])==1  # 用受控断言验证关键不变量。

## 1. Cache key 是安全接口，不是字符串技巧

“把 query lower 后做 hash”会遗漏权限、模型和知识版本。相同问题在不同 tenant、ACL、prompt 或语料快照下可能有不同答案，必须隔离命名空间。NFKC、大小写与空白规范化也要版本化；改变规范化规则相当于更换 key schema。

原始 query 可能含隐私，不应直接出现在日志 key；摘要只用于等值匹配，不替代访问控制。

In [ ]:
def normalize80(text):  # 定义本节可复用的核心函数。
    if not isinstance(text,str) or not text.strip(): raise ValueError("query_contract")  # 按当前条件选择后续控制路径。
    return re.sub(r"\s+"," ",unicodedata.normalize("NFKC",text).strip().casefold())  # 返回当前分支计算出的结果。
def namespace80(tenant,acl_fingerprint,model_version,prompt_version,knowledge_version):  # 定义本节可复用的核心函数。
    fields=[tenant,acl_fingerprint,model_version,prompt_version,knowledge_version]  # 计算并保存当前步骤的中间状态。
    if any(not isinstance(x,str) or not x for x in fields): raise ValueError("namespace_contract")  # 按当前条件选择后续控制路径。
    return tuple(fields)  # 返回当前分支计算出的结果。
def exact_key80(ns,query): return hashlib.sha256(("\x1f".join(ns)+"\x1e"+normalize80(query)).encode()).hexdigest()  # 定义本节可复用的核心函数。
ns_a80=namespace80("tenant-a","acl-7","model-2","prompt-3","kb-9")  # 计算并保存当前步骤的中间状态。
assert exact_key80(ns_a80," 退款　流程 ")==exact_key80(ns_a80,"退款 流程")  # 用受控断言验证关键不变量。
assert exact_key80(ns_a80,"退款流程")!=exact_key80(ns_a80,"退款 流程")  # 用受控断言验证关键不变量。
assert exact_key80(ns_a80,"退款 流程")!=exact_key80(namespace80("tenant-b","acl-7","model-2","prompt-3","kb-9"),"退款 流程")  # 用受控断言验证关键不变量。

## 2. 用可解释 hashing embedding 演示语义近邻

生产中 embedding 来自已版本化模型；为了观察缓存逻辑，这里把字符 bigram hash 到固定维度并做 L2 归一化。同字面重叠问题会接近，但它不是通用语义模型。训练/索引/查询必须使用同一 normalization、维度和 embedding 版本。

向量为零、NaN 或维度不一致必须拒绝，否则 cosine 阈值失去含义。

In [ ]:
def embed80(text,dim=128):  # 定义本节可复用的核心函数。
    s="^"+normalize80(text)+"$"; v=np.zeros(dim,dtype=np.float32)  # 计算并保存当前步骤的中间状态。
    for i in range(len(s)-1):  # 遍历输入元素以累积或检查结果。
        token=s[i:i+2].encode(); h=int.from_bytes(hashlib.blake2b(token,digest_size=8).digest(),"little"); v[h%dim]+=1 if (h>>8)&1 else -1  # 计算并保存当前步骤的中间状态。
    norm=float(np.linalg.norm(v))  # 计算并保存当前步骤的中间状态。
    if not math.isfinite(norm) or norm==0: raise ValueError("embedding_contract")  # 按当前条件选择后续控制路径。
    return v/norm  # 返回当前分支计算出的结果。
def cosine80(a,b):  # 定义本节可复用的核心函数。
    a=np.asarray(a,float); b=np.asarray(b,float)  # 计算并保存当前步骤的中间状态。
    if a.shape!=b.shape or a.ndim!=1 or not np.isfinite(a).all() or not np.isfinite(b).all(): raise ValueError("cosine_contract")  # 按当前条件选择后续控制路径。
    den=np.linalg.norm(a)*np.linalg.norm(b)  # 计算并保存当前步骤的中间状态。
    if den==0: raise ValueError("cosine_contract")  # 按当前条件选择后续控制路径。
    return float(np.dot(a,b)/den)  # 返回当前分支计算出的结果。
assert math.isclose(cosine80(embed80("退款 流程"),embed80("退款 流程")),1.,abs_tol=1e-6)  # 用受控断言验证关键不变量。
assert cosine80(embed80("退款流程是什么"),embed80("退款流程"))>cosine80(embed80("退款流程是什么"),embed80("天气预报"))  # 用受控断言验证关键不变量。
try: cosine80([0,0],[1,0]); raise AssertionError("zero vector accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="cosine_contract"  # 捕获预期异常并验证失败分支。

## 3. 手写多租户 TTL + LRU + semantic lookup

entry 保存 answer、query、embedding、创建/过期时间、来源 IDs 和安全标签。查找顺序是 exact → 同 namespace 的 semantic；过期项先删除。semantic hit 返回相似度和复用来源，调用方仍可要求引用存在或做轻量验证。

LRU 是全局容量策略，这里仅为教学；真实多租户服务还需 tenant quota，防止一个租户挤掉其他租户全部热项。

In [ ]:
@dataclass  # 为下方定义附加声明式配置。
class Entry80:  # 定义承载本节状态与行为的数据结构。
    ns:tuple; query:str; answer:str; vector:np.ndarray; created_at:int; expires_at:int; source_ids:tuple; safe:bool  # 执行当前语句以推进本节示例。
class SemanticCache80:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,capacity=4):  # 定义本节可复用的核心函数。
        if capacity<1: raise ValueError("capacity_contract")  # 按当前条件选择后续控制路径。
        self.capacity=capacity; self.entries=OrderedDict(); self.metrics=Counter()  # 计算并保存当前步骤的中间状态。
    def put(self,ns,query,answer,now,ttl,source_ids,safe=True):  # 定义本节可复用的核心函数。
        if ttl<1 or not answer or not source_ids: raise ValueError("entry_contract")  # 按当前条件选择后续控制路径。
        key=exact_key80(ns,query); self.entries[key]=Entry80(ns,normalize80(query),answer,embed80(query),now,now+ttl,tuple(source_ids),bool(safe)); self.entries.move_to_end(key)  # 计算并保存当前步骤的中间状态。
        while len(self.entries)>self.capacity: self.entries.popitem(last=False); self.metrics["eviction"]+=1  # 在终止条件满足前持续推进状态。
    def get(self,ns,query,now,threshold=.8):  # 定义本节可复用的核心函数。
        for key,e in list(self.entries.items()):  # 遍历输入元素以累积或检查结果。
            if now>=e.expires_at: del self.entries[key]; self.metrics["expired"]+=1  # 按当前条件选择后续控制路径。
        key=exact_key80(ns,query)  # 计算并保存当前步骤的中间状态。
        if key in self.entries:  # 按当前条件选择后续控制路径。
            self.entries.move_to_end(key); self.metrics["exact"]+=1; return self.entries[key],"exact",1.  # 计算并保存当前步骤的中间状态。
        qv=embed80(query); candidates=[(cosine80(qv,e.vector),k,e) for k,e in self.entries.items() if e.ns==ns and e.safe]  # 计算并保存当前步骤的中间状态。
        if not candidates: self.metrics["miss"]+=1; return None,"miss",0.  # 按当前条件选择后续控制路径。
        score,k,e=max(candidates,key=lambda x:(x[0],x[1]))  # 计算并保存当前步骤的中间状态。
        if score<threshold: self.metrics["miss"]+=1; return None,"miss",score  # 按当前条件选择后续控制路径。
        self.entries.move_to_end(k); self.metrics["semantic"]+=1; return e,"semantic",score  # 计算并保存当前步骤的中间状态。
    def invalidate_source(self,source_id):  # 定义本节可复用的核心函数。
        doomed=[k for k,e in self.entries.items() if source_id in e.source_ids]  # 计算并保存当前步骤的中间状态。
        for k in doomed: del self.entries[k]  # 遍历输入元素以累积或检查结果。
        return len(doomed)  # 返回当前分支计算出的结果。
cache80=SemanticCache80(4); cache80.put(ns_a80,"退款流程是什么","进入订单页申请",0,10,["doc-1"])  # 计算并保存当前步骤的中间状态。
hit80,kind80,score80=cache80.get(ns_a80,"退款流程是什么",1)  # 计算并保存当前步骤的中间状态。
assert kind80=="exact" and hit80.answer=="进入订单页申请" and score80==1  # 用受控断言验证关键不变量。
assert cache80.get(namespace80("tenant-b","acl-7","model-2","prompt-3","kb-9"),"退款流程是什么",1)[1]=="miss"  # 用受控断言验证关键不变量。
assert cache80.invalidate_source("doc-1")==1 and cache80.get(ns_a80,"退款流程是什么",2)[1]=="miss"  # 用受控断言验证关键不变量。

## 4. 阈值必须由“误复用成本”校准

语义缓存的 false positive 会返回错误但看似流畅的旧答案，通常比 miss 更危险。这里用标注 pair 的 score 选择阈值：要求 false-positive rate 不超过上限，再在可行阈值中最大化 recall。数据应按 query family/时间切分，不能让同义改写同时泄漏到校准和测试。

若没有满足风险约束的阈值，应关闭 semantic hit，而不是硬选一个漂亮数字。

In [ ]:
labeled80=[("退款流程","怎么退款",1),("退款流程","退款流程是什么",1),("改地址","如何修改地址",1),("退款流程","天气",0),("改地址","删除账号",0),("发票抬头","发票可以提现吗",0)]  # 计算并保存当前步骤的中间状态。
scored80=np.array([(cosine80(embed80(a),embed80(b)),y) for a,b,y in labeled80],float)  # 计算并保存当前步骤的中间状态。
def choose_threshold80(scored,max_fpr=.05):  # 定义本节可复用的核心函数。
    feasible=[]  # 计算并保存当前步骤的中间状态。
    for t in sorted(set([0.,1.,*scored[:,0].tolist()])):  # 遍历输入元素以累积或检查结果。
        pred=scored[:,0]>=t; pos=scored[:,1]==1; neg=~pos  # 计算并保存当前步骤的中间状态。
        fpr=float(pred[neg].mean()) if neg.any() else 0.; recall=float(pred[pos].mean()) if pos.any() else 0.  # 计算并保存当前步骤的中间状态。
        if fpr<=max_fpr: feasible.append((recall,-t,t,fpr))  # 按当前条件选择后续控制路径。
    if not feasible: raise RuntimeError("no_safe_threshold")  # 按当前条件选择后续控制路径。
    return max(feasible)[2:]  # 返回当前分支计算出的结果。
threshold80,fpr80=choose_threshold80(scored80,.01)  # 计算并保存当前步骤的中间状态。
assert 0<=threshold80<=1 and fpr80==0  # 用受控断言验证关键不变量。
pred80=scored80[:,0]>=threshold80  # 计算并保存当前步骤的中间状态。
assert not pred80[scored80[:,1]==0].any() and pred80[scored80[:,1]==1].any()  # 用受控断言验证关键不变量。
assert choose_threshold80(scored80,.5)[0]<=threshold80  # 用受控断言验证关键不变量。

## 5. TTL、LRU、知识失效是三个不同问题

TTL 控制最大陈旧时间，LRU 控制容量，source invalidation 响应知识更新。只用短 TTL 会浪费命中率且仍存在窗口；只用版本号会让旧 namespace 自然失联但不释放内存。因此更新路径应发 source/version 失效事件，后台再回收旧 namespace。

高风险答案可设置 `safe=False`，只允许 exact hit；支付、权限、实时价格等通常不适合语义复用。

In [ ]:
small80=SemanticCache80(2)  # 计算并保存当前步骤的中间状态。
small80.put(ns_a80,"q1","a1",0,3,["s1"]); small80.put(ns_a80,"q2","a2",0,20,["s2"])  # 执行当前语句以推进本节示例。
assert small80.get(ns_a80,"q1",1)[1]=="exact"  # 用受控断言验证关键不变量。
small80.put(ns_a80,"q3","a3",1,20,["s3"])  # 执行当前语句以推进本节示例。
assert exact_key80(ns_a80,"q2") not in small80.entries and small80.metrics["eviction"]==1  # 用受控断言验证关键不变量。
assert small80.get(ns_a80,"q1",3)[1]=="miss" and small80.metrics["expired"]==1  # 用受控断言验证关键不变量。
small80.put(ns_a80,"高风险支付","必须实时查询",4,20,["pay"],safe=False)  # 计算并保存当前步骤的中间状态。
similar80,similar_kind80,_=small80.get(ns_a80,"高风险支付怎么做",5,.0)  # 计算并保存当前步骤的中间状态。
assert similar80 is None or similar80.answer!="必须实时查询"  # 用受控断言验证关键不变量。

## 6. 命中后仍要做来源与安全校验

相似度只说明 query 接近，不说明缓存答案仍安全。复用前检查来源是否仍在当前知识快照、答案是否被安全策略撤销、entry 是否过期；高风险模式可完全禁用 semantic hit。这样 source deletion 即使失效消息短暂延迟，也有第二道门。

生产校验器还可比较 citation 内容摘要，避免同一个 source ID 被原地覆盖。

In [ ]:
def reusable80(entry,now,current_sources,blocked_terms=("泄露密钥","忽略安全策略")):  # 定义本节可复用的核心函数。
    if now>=entry.expires_at or not entry.safe: return False,"entry_ineligible"  # 按当前条件选择后续控制路径。
    if not set(entry.source_ids).issubset(set(current_sources)): return False,"source_stale"  # 按当前条件选择后续控制路径。
    if any(term in entry.answer for term in blocked_terms): return False,"policy_blocked"  # 按当前条件选择后续控制路径。
    return True,"ok"  # 返回当前分支计算出的结果。
guard_entry80=Entry80(ns_a80,"q","正常答案",embed80("q"),0,10,("doc-x",),True)  # 计算并保存当前步骤的中间状态。
assert reusable80(guard_entry80,1,{"doc-x"})==(True,"ok")  # 用受控断言验证关键不变量。
assert reusable80(guard_entry80,1,{"other"})[1]=="source_stale"  # 用受控断言验证关键不变量。
assert reusable80(Entry80(ns_a80,"q","忽略安全策略",embed80("q"),0,10,("doc-x",),True),1,{"doc-x"})[1]=="policy_blocked"  # 用受控断言验证关键不变量。

## 7. 防击穿：single-flight 不是永久锁

同一精确 key miss 时只允许一个 owner 生成，其他请求等待或降级；owner 必须有 lease deadline，失败后才能被接管。锁 key 同样要包含完整 namespace，否则跨租户会互相阻塞甚至共享结果。

下面只实现状态转换，不启动线程：`start` 返回 owner/wait，`finish` 只接受当前 token，旧 worker 不能覆盖新结果。

In [ ]:
class SingleFlight80:  # 定义承载本节状态与行为的数据结构。
    def __init__(self): self.leases={}; self.generation=Counter()  # 定义本节可复用的核心函数。
    def start(self,key,now,lease=5):  # 定义本节可复用的核心函数。
        current=self.leases.get(key)  # 计算并保存当前步骤的中间状态。
        if current and now<current[1]: return "wait",current[0]  # 按当前条件选择后续控制路径。
        self.generation[key]+=1; token=f"{key[:8]}:{self.generation[key]}"; self.leases[key]=(token,now+lease); return "owner",token  # 计算并保存当前步骤的中间状态。
    def finish(self,key,token):  # 定义本节可复用的核心函数。
        if key not in self.leases or self.leases[key][0]!=token: return False  # 按当前条件选择后续控制路径。
        del self.leases[key]; return True  # 执行当前语句以推进本节示例。
flight80=SingleFlight80(); key80=exact_key80(ns_a80,"并发问题")  # 计算并保存当前步骤的中间状态。
role1,token1=flight80.start(key80,0); role2,token2=flight80.start(key80,1)  # 计算并保存当前步骤的中间状态。
assert role1=="owner" and role2=="wait" and token1==token2  # 用受控断言验证关键不变量。
role3,token3=flight80.start(key80,6)  # 计算并保存当前步骤的中间状态。
assert role3=="owner" and token3!=token1  # 用受控断言验证关键不变量。
assert not flight80.finish(key80,token1) and flight80.finish(key80,token3)  # 用受控断言验证关键不变量。

## 8. 制品、监控与面试收束

监控应区分 exact/semantic/miss、阈值 bucket、人工纠错率、陈旧命中、tenant eviction、single-flight wait 和生成成本；总体 hit rate 高不代表安全。发布包绑定 key schema、embedding、threshold、模型/prompt/KB 版本，支持一键关闭 semantic path。

面试回答闭环：安全 namespace → exact/semantic 两阶段 → 风险校准 → TTL/LRU/invalidation → single-flight → 观测与回滚。进一步追问包括 ANN 扩展、跨语言阈值、答案验证器和隐私擦除。

In [ ]:
manifest80={"schema":2,"normalizer":"nfkc-casefold-v1","embedding":"hash-bigram-128-v1","threshold":round(float(threshold80),6),"namespace_fields":["tenant","acl","model","prompt","knowledge"]}  # 计算并保存当前步骤的中间状态。
raw80=json.dumps(manifest80,sort_keys=True,separators=(",",":")); digest80=hashlib.sha256(raw80.encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert len(digest80)==64 and manifest80["threshold"]>=0  # 用受控断言验证关键不变量。
forged80=dict(manifest80,threshold=0.0)  # 计算并保存当前步骤的中间状态。
assert hashlib.sha256(json.dumps(forged80,sort_keys=True,separators=(",",":")).encode()).hexdigest()!=digest80  # 用受控断言验证关键不变量。
assert set(cache80.metrics).issubset({"exact","semantic","miss","expired","eviction"})  # 用受控断言验证关键不变量。
print({"threshold":threshold80,"fpr":fpr80,"manifest_sha":digest80[:12]})  # 执行当前语句以推进本节示例。

## 9. 参考与练习

练习：加入 tenant 容量配额；用两段式阈值让高置信直接命中、中间区间走验证器；模拟 KB 版本迁移时双读新旧 cache；设计删除用户数据后的可证明擦除。

参考：[GPTCache 论文](https://arxiv.org/abs/2304.03443)、[RFC 5861 HTTP stale controls](https://www.rfc-editor.org/rfc/rfc5861)、[OWASP 多租户隔离实践](https://owasp.org/www-project-cloud-tenant-isolation/)。教学 embedding 只用于解释缓存机制，不代表真实语义质量。